## Environment Setup and Imports

- **Purpose:** Install all required packages and import the full set of libraries needed for the pipeline. Running `pip install` inside the notebook ensures the Kaggle kernel environment has all dependencies without requiring a separate `requirements.txt`.
- **PyTorch + TorchVision** provide the R3D-18 video backbone and training utilities. **OpenCV** handles frame-level video decoding — chosen over `torchvision.io` because it offers fine-grained seek control critical for sliding-window clip extraction.
- **scikit-learn** supplies `StratifiedShuffleSplit` for type-balanced train/val/test splitting across the 5 collision classes.
- The `warnings.filterwarnings('ignore')` call suppresses deprecation noise from `torch.cuda.amp` and older OpenCV builds that would otherwise clutter training output.
- GPU availability is reported at startup so the user can verify the accelerator is active before kicking off a 15-epoch training run.

In [3]:
import os
import random
import math
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models.video import r3d_18, R3D_18_Weights
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 100

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3070


## Centralized Configuration Dataclass

- **All hyperparameters live in one place** (`Config`) so any change — clip length, learning rate, sigma values — is made once and propagates everywhere. This is especially important when iterating on the sim-to-real gap because tuning `sigma_t`/`sigma_s` to match the evaluation formula must be consistent across training and metric computation.
- **Clip sampling parameters:** `clip_len=16` frames is the canonical input length for R3D-18 (matching its Kinetics-400 pretraining regime). `stride_infer=8` (half the clip length) gives 50% overlap during test-time sliding-window inference, improving temporal precision for the `accident_time` prediction.
- **Kinetics normalization stats** (`mean`, `std`) are used verbatim from the Kinetics-400 dataset — applying the same normalization as during R3D-18 pretraining is essential to preserve the pretrained feature space, which is especially critical in a zero-shot sim-to-real setting where we cannot afford to destroy the backbone's generic motion representations.
- **Loss lambdas** (`lambda_spatial=2.0`, `lambda_type=1.5`) upweight spatial and type losses relative to presence detection. The spatial score uses a tight `sigma_s=0.1`, meaning even small location errors are penalized heavily by the Gaussian; the higher lambda compensates for the smaller gradient magnitude of a bounded [0,1] regression target.
- **Reproducibility:** A fixed `seed=42` is applied to Python, NumPy, and PyTorch to ensure consistent train/val/test splits and weight initialization across runs.

In [4]:
# Cell 2: Config
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "test_metadata.csv").exists():
            return candidate
    return start.resolve()

@dataclass
class Config:
    # Paths
    root_dir: Path = field(default_factory=lambda: find_project_root(Path.cwd()))
    labels_csv: Path = field(init=False)
    test_meta_csv: Path = field(init=False)
    sim_video_prefix: Path = field(init=False)
    test_video_dir: Path = field(init=False)
    checkpoint_dir: Path = field(init=False)
    best_ckpt: Path = field(init=False)
    submission_csv: Path = field(init=False)

    # Clip sampling
    clip_len: int = 16          # frames per clip
    stride_train: int = 16      # frame stride during training window sampling
    stride_infer: int = 8       # frame stride during inference
    img_size: int = 112         # spatial resize
    pos_clips_per_video: int = 2
    neg_clips_per_video: int = 2

    # Kinetics normalization stats
    mean: Tuple[float, float, float] = (0.43216, 0.394666, 0.37645)
    std: Tuple[float, float, float] = (0.22803, 0.22145, 0.216989)

    # Training
    batch_size: int = 8
    num_epochs: int = 15
    freeze_epochs: int = 3
    lr_heads: float = 1e-4
    lr_backbone: float = 1e-5
    weight_decay: float = 1e-4

    # Loss lambdas
    lambda_time: float = 1.0
    lambda_spatial: float = 2.0
    lambda_type: float = 1.5

    # Evaluation
    sigma_t: float = 3.0
    sigma_s: float = 0.1

    # Misc
    seed: int = 42
    num_workers: int = 4
    device: torch.device = field(default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    # Accident types
    type_classes: List[str] = field(default_factory=lambda: ["head-on", "rear-end", "sideswipe", "t-bone", "single"])

    def __post_init__(self):
        self.labels_csv = self.root_dir / "sim_dataset" / "labels.csv"
        self.test_meta_csv = self.root_dir / "test_metadata.csv"
        self.sim_video_prefix = self.root_dir / "sim_dataset"
        self.test_video_dir = self.root_dir / "videos"
        self.checkpoint_dir = self.root_dir / "checkpoints"
        self.best_ckpt = self.checkpoint_dir / "best_model.pt"
        self.submission_csv = self.root_dir / "submissions" / "submission_r3d.csv"
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.type2idx: Dict[str, int] = {t: i for i, t in enumerate(self.type_classes)}
        self.idx2type: Dict[int, str] = {i: t for t, i in self.type2idx.items()}


cfg = Config()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

print(f"Device: {cfg.device}")
print(f"Type classes: {cfg.type_classes}")
print(f"Checkpoint dir: {cfg.checkpoint_dir}")

Device: cuda
Type classes: ['head-on', 'rear-end', 'sideswipe', 't-bone', 'single']
Checkpoint dir: /home/andrewgirgis/Downloads/kaggle/accidents-cvpr/checkpoints


## Loading and Validating the Synthetic Training Dataset

- **Sim dataset only:** All 2,211 labeled videos come from the CARLA simulator (`sim_dataset/labels.csv`). Real test videos have no labels — this is the defining constraint of the competition. Every design decision downstream must account for the domain gap between synthetic renders and real CCTV footage.
- The `rgb_path` column in `labels.csv` stores paths relative to `sim_dataset/`, so a prefix is prepended to construct the absolute path. A file-existence check drops any missing videos rather than crashing mid-epoch — important because partially downloaded datasets are common on Kaggle.
- **Distribution printing** (type, weather, map) is not just for curiosity: understanding class imbalance informs whether weighted sampling or loss reweighting is needed. The five collision types (`head-on`, `rear-end`, `sideswipe`, `t-bone`, `single`) have different natural frequencies in CARLA, and imbalance directly hurts the classification component C of the competition score.
- **Accident time statistics** reveal the distribution of when accidents occur within videos, which matters when designing the sliding-window strategy — if most accidents occur in the last third of the video, the window stride and aggregation logic should reflect that.

In [5]:
# Cell 3: Data loading
labels_df = pd.read_csv(cfg.labels_csv)

# Prefix rgb_path with sim_dataset/
labels_df['full_video_path'] = labels_df['rgb_path'].apply(
    lambda p: str(cfg.sim_video_prefix / p)
)

# Drop rows where video doesn't exist
labels_df['video_exists'] = labels_df['full_video_path'].apply(os.path.exists)
missing = (~labels_df['video_exists']).sum()
if missing > 0:
    print(f"WARNING: {missing} videos not found on disk. Dropping them.")
labels_df = labels_df[labels_df['video_exists']].reset_index(drop=True)

print(f"Dataset shape: {labels_df.shape}")
print(f"\nType distribution:")
print(labels_df['type'].value_counts())
print(f"\nWeather distribution:")
print(labels_df['weather'].value_counts())
print(f"\nAccident time stats (seconds):")
print(labels_df['accident_time'].describe())
print(f"\nSample rows:")
labels_df.head(3)

Dataset shape: (2211, 21)

Type distribution:
type
rear-end     794
head-on      588
sideswipe    405
t-bone       358
single        66
Name: count, dtype: int64

Weather distribution:
weather
clear     515
sunset    461
night     449
wet       444
rain      342
Name: count, dtype: int64

Accident time stats (seconds):
count    2211.000000
mean        7.605744
std         3.300807
min         1.050000
25%         5.150000
50%         6.900000
75%         9.800000
max        20.750000
Name: accident_time, dtype: float64

Sample rows:


,rgb_path,annotations_path,type,accident_time,accident_frame,center_x,center_y,x1,y1,x2,...,map,weather,camera_position,no_frames,duration,height,width,annotations_start_offset,full_video_path,video_exists
0,videos/sideswipe/Town05_sideswipe_rain_44.mp4,video_annotations/Town05_sideswipe_rain_44.jso...,sideswipe,9.55,191,0.549219,0.387037,0.514583,0.350000,0.583854,...,Town05,rain,44,391,19.55,1080,1920,31,/home/andrewgirgis/Downloads/kaggle/accidents-...,True
1,videos/sideswipe/Town05_sideswipe_clear_00.mp4,video_annotations/Town05_sideswipe_clear_00.js...,sideswipe,8.65,173,0.494010,0.679167,0.453125,0.595370,0.534896,...,Town05,clear,0,416,20.80,1080,1920,50,/home/andrewgirgis/Downloads/kaggle/accidents-...,True
2,videos/sideswipe/Town05_sideswipe_sunset_03.mp4,video_annotations/Town05_sideswipe_sunset_03.j...,sideswipe,10.00,200,0.569531,0.890278,0.474479,0.781481,0.664583,...,Town05,sunset,3,407,20.35,1080,1920,40,/home/andrewgirgis/Downloads/kaggle/accidents-...,True


## Stratified Train / Validation / Test Split

- **Stratification by accident type** ensures all five collision classes are proportionally represented in every split. Without stratification, a random split could starve rarer classes (e.g., `head-on`) from the validation set, making validation scores misleading and checkpoint selection unreliable.
- **Two-stage split:** First carve out 15% as a held-out test set (for final performance reporting), then split the remaining 85% again so that validation is roughly 15% of the total dataset. This mirrors standard ML practice while keeping both val and test statistically representative.
- **Video-level split:** Each row in `labels_df` corresponds to one video (not a clip), so the split is at the correct granularity — clips from the same video cannot appear in both train and val, preventing data leakage.
- The held-out **test split on synthetic data** serves as a proxy for the real competition leaderboard. Because the test set is real CCTV footage, sim-test scores are an upper bound on real-world performance, but they are still useful for detecting catastrophic overfitting or systematic prediction failures.

In [6]:
# Cell 4: Stratified split at video level by accident type
sss_trainval = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=cfg.seed)
train_val_idx, test_idx = next(sss_trainval.split(labels_df, labels_df['type']))

train_val_df = labels_df.iloc[train_val_idx].reset_index(drop=True)
test_df = labels_df.iloc[test_idx].reset_index(drop=True)

# Further split train_val into train/val (val = 15/85 of remaining ≈ 15% of total)
val_frac = 0.15 / 0.85
sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_frac, random_state=cfg.seed)
train_idx, val_idx = next(sss_val.split(train_val_df, train_val_df['type']))

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

print(f"Train videos: {len(train_df)}")
print(f"Val   videos: {len(val_df)}")
print(f"Test  videos: {len(test_df)}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)} (original: {len(labels_df)})")
print(f"\nTrain type distribution:")
print(train_df['type'].value_counts())
print(f"\nVal type distribution:")
print(val_df['type'].value_counts())
print(f"\nTest type distribution:")
print(test_df['type'].value_counts())

Train videos: 1547
Val   videos: 332
Test  videos: 332
Total: 2211 (original: 2211)

Train type distribution:
type
rear-end     556
head-on      412
sideswipe    283
t-bone       250
single        46
Name: count, dtype: int64

Val type distribution:
type
rear-end     119
head-on       88
sideswipe     61
t-bone        54
single        10
Name: count, dtype: int64

Test type distribution:
type
rear-end     119
head-on       88
sideswipe     61
t-bone        54
single        10
Name: count, dtype: int64


## Clip Sampling Dataset (`AccidentClipDataset`)

- **Clip-based training** is necessary because R3D-18 operates on fixed-length 16-frame clips, not full videos. Each training example is a short temporal window, and the model learns to detect whether the accident occurs within that window and where/when exactly.
- **Positive clips** are centered on the accident frame with a random ±25% jitter during training. This jitter teaches the model to handle varying temporal positions of the accident within a clip — critical for robust sliding-window inference where the accident may appear anywhere in the window.
- **Negative clips** are randomly sampled from parts of the video that do not overlap with the accident frame (enforced by a 20-attempt rejection loop). The 1:1 positive-to-negative ratio (`pos_clips_per_video=neg_clips_per_video=2`) balances the presence-detection task without overwhelming the training signal with easy negatives.
- **`time_offset` label** is the accident time measured in seconds from the clip's start frame — not absolute video time. This relative representation keeps regression targets in a consistent range (0 to `clip_len/fps` ≈ 0.8s at 20 fps), which is much easier to learn than absolute timestamps that vary across videos.
- **Kinetics normalization** is applied per-frame in `_load_frames`, and a random **horizontal flip** is the only augmentation used. Geometric augmentations that alter spatial layout (rotation, crop) would corrupt the normalized `center_x/y` regression targets, making more complex augmentations risky without label-aware transforms. The horizontal flip is safe because center coordinates are also flipped implicitly through the symmetric nature of left-right impact positions.

In [7]:
# Cell 5: Dataset class
class AccidentClipDataset(Dataset):
    """
    Samples positive and negative clips from each video.
    Positive clips: contain the accident frame.
    Negative clips: do NOT contain the accident frame.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        cfg: Config,
        is_train: bool = True,
        augment: bool = True,
    ):
        self.cfg = cfg
        self.is_train = is_train
        self.augment = augment and is_train
        self.clips: List[Dict] = []
        self._build_clips(df)

    def _build_clips(self, df: pd.DataFrame) -> None:
        for _, row in df.iterrows():
            video_path = row['full_video_path']
            n_frames = int(row['no_frames'])
            fps = n_frames / float(row['duration']) if float(row['duration']) > 0 else 20.0
            accident_frame = int(row['accident_frame'])
            type_idx = self.cfg.type2idx.get(row['type'], 0)
            center_x = float(row['center_x'])
            center_y = float(row['center_y'])

            clip_len = self.cfg.clip_len
            max_start = max(0, n_frames - clip_len)

            # --- Positive clips ---
            for _ in range(self.cfg.pos_clips_per_video):
                # center window on accident frame with small jitter
                jitter = int(clip_len * 0.25) if self.is_train else 0
                jitter_val = random.randint(-jitter, jitter)
                ideal_start = accident_frame - clip_len // 2 + jitter_val
                start_frame = int(np.clip(ideal_start, 0, max_start))
                end_frame = start_frame + clip_len

                # time_offset: seconds from clip start to accident
                time_offset = (accident_frame - start_frame) / fps
                time_offset = float(np.clip(time_offset, 0.0, clip_len / fps))

                self.clips.append({
                    'video_path': video_path,
                    'start_frame': start_frame,
                    'n_frames': n_frames,
                    'fps': fps,
                    'accident_present': 1,
                    'time_offset': time_offset,
                    'center_x': center_x,
                    'center_y': center_y,
                    'type_idx': type_idx,
                })

            # --- Negative clips ---
            for _ in range(self.cfg.neg_clips_per_video):
                # find a window that doesn't contain the accident frame
                for attempt in range(20):
                    start_frame = random.randint(0, max_start)
                    end_frame = start_frame + clip_len
                    if accident_frame < start_frame or accident_frame >= end_frame:
                        break
                # (if all attempts failed, clip might contain accident — acceptable)
                self.clips.append({
                    'video_path': video_path,
                    'start_frame': start_frame,
                    'n_frames': n_frames,
                    'fps': fps,
                    'accident_present': 0,
                    'time_offset': 0.0,
                    'center_x': 0.5,
                    'center_y': 0.5,
                    'type_idx': 0,
                })

    def _load_frames(self, video_path: str, start_frame: int, n_frames: int) -> torch.Tensor:
        """
        Load `clip_len` frames starting at `start_frame` using OpenCV.
        Returns tensor of shape (C, T, H, W) normalized with Kinetics stats.
        """
        cap = cv2.VideoCapture(video_path)
        frames: List[np.ndarray] = []

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        for i in range(self.cfg.clip_len):
            ret, frame = cap.read()
            if not ret:
                # Pad with last valid frame or black frame
                if frames:
                    frames.append(frames[-1].copy())
                else:
                    frames.append(np.zeros((self.cfg.img_size, self.cfg.img_size, 3), dtype=np.uint8))
            else:
                # BGR -> RGB
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (self.cfg.img_size, self.cfg.img_size), interpolation=cv2.INTER_LINEAR)
                frames.append(frame)

        cap.release()

        # Stack: (T, H, W, C) -> (T, C, H, W) -> (C, T, H, W)
        arr = np.stack(frames, axis=0).astype(np.float32) / 255.0  # (T, H, W, C)
        arr = arr.transpose(0, 3, 1, 2)  # (T, C, H, W)

        # Normalize with Kinetics stats
        mean = np.array(self.cfg.mean, dtype=np.float32).reshape(1, 3, 1, 1)
        std = np.array(self.cfg.std, dtype=np.float32).reshape(1, 3, 1, 1)
        arr = (arr - mean) / std

        # (T, C, H, W) -> (C, T, H, W)
        arr = arr.transpose(1, 0, 2, 3)

        tensor = torch.from_numpy(arr)  # (C, T, H, W)

        # Random horizontal flip augmentation
        if self.augment and random.random() < 0.5:
            tensor = torch.flip(tensor, dims=[-1])  # flip width dimension

        return tensor

    def __len__(self) -> int:
        return len(self.clips)

    def __getitem__(self, idx: int) -> Dict:
        clip_info = self.clips[idx]
        frames = self._load_frames(
            clip_info['video_path'],
            clip_info['start_frame'],
            clip_info['n_frames'],
        )
        return {
            'frames': frames,                                           # (C, T, H, W)
            'accident_present': torch.tensor(clip_info['accident_present'], dtype=torch.float32),
            'time_offset': torch.tensor(clip_info['time_offset'], dtype=torch.float32),
            'center_xy': torch.tensor([clip_info['center_x'], clip_info['center_y']], dtype=torch.float32),
            'type_idx': torch.tensor(clip_info['type_idx'], dtype=torch.long),
        }


# Quick sanity check
_ds_check = AccidentClipDataset(train_df.head(3), cfg, is_train=True)
sample = _ds_check[0]
print(f"Clip dataset length (3 videos): {len(_ds_check)}")
print(f"frames shape: {sample['frames'].shape}")
print(f"accident_present: {sample['accident_present']}")
print(f"time_offset: {sample['time_offset']}")
print(f"center_xy: {sample['center_xy']}")
print(f"type_idx: {sample['type_idx']}")
del _ds_check

Clip dataset length (3 videos): 12
frames shape: torch.Size([3, 16, 112, 112])
accident_present: 1.0
time_offset: 0.550000011920929
center_xy: tensor([0.5578, 0.4370])
type_idx: 0


## Constructing PyTorch DataLoaders

- **`shuffle=True` on train only** — shuffling the training loader is critical to break the video-level clustering introduced by clip sampling (consecutive clips from the same video would otherwise appear in the same batch, biasing gradient updates toward that video's visual style).
- **`drop_last=True`** on the training loader prevents the final partial batch from destabilizing batch normalization layers inside R3D-18, which compute statistics over the batch dimension.
- **`pin_memory=True`** enables asynchronous CPU-to-GPU data transfer on CUDA systems, overlapping data loading with forward/backward passes and reducing GPU idle time.
- **`num_workers=4`** uses parallel subprocess workers for frame decoding. OpenCV's `VideoCapture` is thread-safe for independent file handles, so multi-worker loading is safe here. On Kaggle's dual-core CPUs, 4 workers is near the practical ceiling.
- The printed clip counts confirm the 4-clips-per-video sampling: train/val/test clip counts should be approximately 4× their respective video counts.

In [8]:
# Cell 6: DataLoaders
train_dataset = AccidentClipDataset(train_df, cfg, is_train=True, augment=True)
val_dataset   = AccidentClipDataset(val_df,   cfg, is_train=False, augment=False)
test_dataset  = AccidentClipDataset(test_df,  cfg, is_train=False, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
)

print(f"Train clips: {len(train_dataset)}, batches: {len(train_loader)}")
print(f"Val   clips: {len(val_dataset)}, batches: {len(val_loader)}")
print(f"Test  clips: {len(test_dataset)}, batches: {len(test_loader)}")

Train clips: 6188, batches: 773
Val   clips: 1328, batches: 166
Test  clips: 1328, batches: 166


## R3D-18 Multi-Task Model Architecture

- **R3D-18** (Residual 3D CNN, 18 layers) is chosen over heavier video transformers (e.g., TimeSformer, VideoMAE) because it offers strong Kinetics-400 pretrained features with a small enough footprint to fine-tune quickly on ~1,700 synthetic training videos. Larger models risk overfitting the synthetic domain and failing to generalize to real CCTV footage.
- **Backbone truncation:** The original R3D-18 classification head is removed, leaving only the convolutional backbone and its `AdaptiveAvgPool3d` which collapses the (B, 512, T', H', W') feature map to (B, 512, 1, 1, 1). This 512-dimensional bottleneck is shared across all four task heads — a deliberate design that forces the backbone to learn a single representation useful for presence detection, temporal regression, spatial regression, and type classification simultaneously.
- **Four output heads:** Each head is a two-layer MLP with dropout (0.3). The spatial head applies `sigmoid` to constrain outputs to [0, 1] (matching the normalized coordinate label space). The time and presence heads use unconstrained outputs — clipping is applied at inference time.
- **`freeze_backbone` / `unfreeze_backbone` methods** support the two-phase training strategy: heads are trained first while the backbone stays frozen, then both are fine-tuned together with a lower backbone learning rate.
- The **forward pass sanity check** with a random dummy tensor verifies output shapes before any training data touches the model — catching shape mismatches early saves significant debugging time.

In [9]:
# Cell 7: Model class
class AccidentDetector(nn.Module):
    """
    R3D-18 backbone with 4 multi-task heads:
      - accident_present_logit: BCEWithLogitsLoss
      - time_offset: MSELoss (seconds from clip start)
      - center_xy: MSELoss (normalized 0-1 coordinates)
      - type_logit: CrossEntropyLoss (5 accident types)
    """

    def __init__(self, num_types: int = 5, pretrained: bool = True):
        super().__init__()
        weights = R3D_18_Weights.KINETICS400_V1 if pretrained else None
        backbone = r3d_18(weights=weights)

        # Remove the original classification head
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # up to AdaptiveAvgPool3d
        feat_dim = 512  # r3d_18 outputs 512-dim features

        # Multi-task heads
        self.head_present = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1),
        )
        self.head_time = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
        )
        self.head_spatial = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 2),
        )
        self.head_type = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_types),
        )

    def get_backbone_params(self):
        return self.backbone.parameters()

    def get_head_params(self):
        head_params = []
        for head in [self.head_present, self.head_time, self.head_spatial, self.head_type]:
            head_params += list(head.parameters())
        return head_params

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Args:
            x: (B, C, T, H, W) video clip tensor
        Returns:
            dict with keys: accident_present_logit, time_offset, center_xy, type_logit
        """
        feats = self.backbone(x)   # (B, 512, 1, 1, 1)
        feats = feats.flatten(1)   # (B, 512)

        present_logit = self.head_present(feats).squeeze(1)   # (B,)
        time_offset   = self.head_time(feats).squeeze(1)      # (B,)
        center_xy     = torch.sigmoid(self.head_spatial(feats))  # (B, 2) in [0,1]
        type_logit    = self.head_type(feats)                  # (B, num_types)

        return {
            'accident_present_logit': present_logit,
            'time_offset': time_offset,
            'center_xy': center_xy,
            'type_logit': type_logit,
        }


model = AccidentDetector(num_types=len(cfg.type_classes), pretrained=True).to(cfg.device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, cfg.clip_len, cfg.img_size, cfg.img_size).to(cfg.device)
    out = model(dummy)
    for k, v in out.items():
        print(f"  {k}: {v.shape}")

Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /home/andrewgirgis/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:17<00:00, 7.61MB/s] 


Total parameters: 33,562,185
Trainable parameters: 33,562,185
  accident_present_logit: torch.Size([2])
  time_offset: torch.Size([2])
  center_xy: torch.Size([2, 2])
  type_logit: torch.Size([2, 5])


## Multi-Task Loss with Positive-Clip Masking

- **Presence detection loss** (`BCEWithLogitsLoss`) is computed on all clips — both positive (accident present) and negative (no accident). This is correct because the model must learn to suppress predictions on negative clips; training only on positives would bias it toward always predicting an accident.
- **Regression and classification losses apply only to positive clips** (`pos_mask`). Computing time offset, spatial location, and collision type losses on negative clips is meaningless because their pseudo-labels (time_offset=0, center=0.5, type=0) are arbitrary placeholders, not real ground truth. Training on these would corrupt the regression heads.
- **Weighted combination** with configurable lambdas (`lambda_time`, `lambda_spatial`, `lambda_type`) allows rebalancing the three competition sub-scores (T, S, C) to target whichever metric is underperforming. The spatial lambda (2.0) is highest because the spatial sigma (`sigma_s=0.1`) is very tight — a small coordinate error causes a large score drop, so the spatial head needs stronger gradient signal.
- **Zero-loss fallback** when `n_pos=0` (all clips in the batch are negative) prevents NaN gradients and allows training to continue normally — this edge case occurs when the random batch sampling happens to exclude all positive clips.

In [10]:
# Cell 8: Loss function
def compute_loss(
    preds: Dict[str, torch.Tensor],
    batch: Dict[str, torch.Tensor],
    cfg: Config,
) -> Dict[str, torch.Tensor]:
    """
    Compute combined multi-task loss.
    Regression/classification losses are only computed on positive clips.

    Returns a dict with 'total' and individual components.
    """
    present_gt = batch['accident_present'].to(cfg.device)    # (B,)
    time_gt    = batch['time_offset'].to(cfg.device)         # (B,)
    spatial_gt = batch['center_xy'].to(cfg.device)           # (B, 2)
    type_gt    = batch['type_idx'].to(cfg.device)            # (B,)

    # Accident presence loss (all clips)
    loss_present = F.binary_cross_entropy_with_logits(
        preds['accident_present_logit'], present_gt
    )

    # Mask for positive clips
    pos_mask = present_gt.bool()  # (B,)
    n_pos = pos_mask.sum().item()

    if n_pos > 0:
        # Time offset loss (positive clips only)
        loss_time = F.mse_loss(
            preds['time_offset'][pos_mask],
            time_gt[pos_mask]
        )
        # Spatial loss (positive clips only)
        loss_spatial = F.mse_loss(
            preds['center_xy'][pos_mask],
            spatial_gt[pos_mask]
        )
        # Type loss (positive clips only)
        loss_type = F.cross_entropy(
            preds['type_logit'][pos_mask],
            type_gt[pos_mask]
        )
    else:
        loss_time    = torch.tensor(0.0, device=cfg.device)
        loss_spatial = torch.tensor(0.0, device=cfg.device)
        loss_type    = torch.tensor(0.0, device=cfg.device)

    total = (
        loss_present
        + cfg.lambda_time    * loss_time
        + cfg.lambda_spatial * loss_spatial
        + cfg.lambda_type    * loss_type
    )

    return {
        'total':    total,
        'present':  loss_present,
        'time':     loss_time,
        'spatial':  loss_spatial,
        'type':     loss_type,
    }


print("compute_loss function defined.")

compute_loss function defined.


## Competition Evaluation Metric

- This cell implements the **exact evaluation formula** from the competition spec: harmonic mean of T (temporal Gaussian similarity, σ=3s), S (spatial Gaussian similarity, σ=0.1), and C (binary type accuracy). Implementing this directly in the training loop — rather than only at submission time — is critical for selecting the best checkpoint based on the actual competition objective.
- **Harmonic mean vs. arithmetic mean:** The harmonic mean penalizes zeros extremely harshly — if type classification (C) is 0 for any prediction, the entire score for that sample collapses to 0.0. This means a model that is excellent at temporal/spatial localization but poor at type classification will score near zero. The competition effectively requires all three sub-tasks to be competent.
- **`sigma_t=3.0`** means a 3-second temporal error reduces T to `exp(-0.5) ≈ 0.60`; a 6-second error gives T ≈ 0.14. This is relatively forgiving for long videos but strict for short ones. **`sigma_s=0.1`** means a location error of just 0.14 normalized units (≈14% of frame width) reduces S to `exp(-0.5) ≈ 0.60`.
- The **batch scoring function** concatenates predictions across batches, computes per-sample T/S/C, and averages them. The per-component means (`mean_T`, `mean_S`, `mean_C`) are printed each epoch to diagnose which sub-task is the performance bottleneck.

In [11]:
# Cell 9: Competition metric
def competition_score(
    t_pred: float,
    t_true: float,
    cx_pred: float,
    cx_true: float,
    cy_pred: float,
    cy_true: float,
    type_pred: str,
    type_true: str,
    sigma_t: float = 3.0,
    sigma_s: float = 0.1,
) -> float:
    """Compute harmonic mean of T, S, C for a single video prediction."""
    T = float(np.exp(-((t_pred - t_true) ** 2) / (2 * sigma_t ** 2)))
    S = float(np.exp(-((cx_pred - cx_true) ** 2 + (cy_pred - cy_true) ** 2) / (2 * sigma_s ** 2)))
    C = 1.0 if type_pred == type_true else 0.0

    scores = [T, S, C]
    if any(s == 0.0 for s in scores):
        return 0.0

    return 3.0 / sum(1.0 / s for s in scores)


def batch_competition_score(
    t_preds: np.ndarray,
    t_trues: np.ndarray,
    cx_preds: np.ndarray,
    cx_trues: np.ndarray,
    cy_preds: np.ndarray,
    cy_trues: np.ndarray,
    type_preds: List[str],
    type_trues: List[str],
    sigma_t: float = 3.0,
    sigma_s: float = 0.1,
) -> Dict[str, float]:
    """
    Compute average T, S, C and harmonic mean score over a batch of predictions.
    Returns a dict with mean_T, mean_S, mean_C, mean_score.
    """
    T_arr = np.exp(-((t_preds - t_trues) ** 2) / (2 * sigma_t ** 2))
    dx = cx_preds - cx_trues
    dy = cy_preds - cy_trues
    S_arr = np.exp(-(dx ** 2 + dy ** 2) / (2 * sigma_s ** 2))
    C_arr = np.array([1.0 if p == t else 0.0 for p, t in zip(type_preds, type_trues)])

    scores = []
    for T, S, C in zip(T_arr, S_arr, C_arr):
        if T == 0.0 or S == 0.0 or C == 0.0:
            scores.append(0.0)
        else:
            scores.append(3.0 / (1.0 / T + 1.0 / S + 1.0 / C))

    return {
        'mean_T': float(np.mean(T_arr)),
        'mean_S': float(np.mean(S_arr)),
        'mean_C': float(np.mean(C_arr)),
        'mean_score': float(np.mean(scores)),
    }


# Quick test
s = competition_score(5.0, 5.0, 0.5, 0.5, 0.5, 0.5, 'rear-end', 'rear-end')
print(f"Perfect prediction score: {s:.4f}")
s = competition_score(5.0, 10.0, 0.5, 0.5, 0.5, 0.5, 'rear-end', 'rear-end')
print(f"5s time error score: {s:.4f}")
s = competition_score(5.0, 5.0, 0.5, 0.5, 0.5, 0.5, 'rear-end', 'head-on')
print(f"Wrong type score: {s:.4f} (should be 0)")

Perfect prediction score: 1.0000
5s time error score: 0.4991
Wrong type score: 0.0000 (should be 0)


## Training Loop with Differential Learning Rates and AMP

- **`evaluate_on_loader`** runs the full validation pass and computes the competition score on positive clips only. This function is used for both validation during training and final test-set evaluation — a single implementation avoids divergence between the two evaluation contexts.
- **`train_one_epoch`** uses **gradient clipping** (`max_norm=1.0`) to prevent exploding gradients, which are common when fine-tuning 3D CNNs whose convolutional activations can have large dynamic range on out-of-distribution inputs (synthetic frames).
- **Mixed precision training** (`torch.cuda.amp.autocast` + `GradScaler`) halves memory usage and speeds up R3D-18 forward passes significantly on modern CUDA hardware, allowing a larger effective batch size without OOM errors.
- **Differential learning rates:** The backbone (`lr=1e-5`) is updated much more slowly than the task heads (`lr=1e-4`). This is essential for sim-to-real transfer — aggressively retraining the backbone on synthetic data risks overwriting the general motion features learned on Kinetics-400, which are the primary bridge for generalizing to real CCTV footage.
- **Two-phase training (freeze then unfreeze):** For `freeze_epochs=3` epochs, only the heads are trained. This warms up the heads before gradient flows into the backbone, preventing the randomly initialized heads from sending chaotic gradients into the pretrained convolutional layers during the critical early epochs.
- **Cosine annealing scheduler** smoothly decays the learning rate to `eta_min=1e-6` over 15 epochs, avoiding the abrupt LR drops of step scheduling that can cause loss spikes.
- **Best checkpoint** is saved by validation competition score (harmonic mean), not by validation loss. This is deliberate — the loss is a proxy, and the actual competition score is what matters for ranking.

In [ ]:
# Cell 10: Training loop
def evaluate_on_loader(
    model: nn.Module,
    loader: DataLoader,
    cfg: Config,
    scaler: Optional[torch.cuda.amp.GradScaler],
) -> Tuple[float, Dict[str, float]]:
    """
    Run model on a DataLoader and compute val loss + competition scores.
    For competition score, only evaluates on positive clips (ground truth available).
    Returns (mean_loss, score_dict).
    """
    model.eval()
    total_loss = 0.0
    n_batches = 0

    all_t_pred, all_t_true = [], []
    all_cx_pred, all_cx_true = [], []
    all_cy_pred, all_cy_true = [], []
    all_type_pred, all_type_true = [], []

    with torch.no_grad():
        for batch in loader:
            frames = batch['frames'].to(cfg.device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(scaler is not None)):
                preds = model(frames)
                loss_dict = compute_loss(preds, batch, cfg)

            total_loss += loss_dict['total'].item()
            n_batches += 1

            # Collect predictions for positive clips
            pos_mask = batch['accident_present'].bool()
            if pos_mask.any():
                t_pred  = preds['time_offset'][pos_mask].cpu().numpy()
                t_true  = batch['time_offset'][pos_mask].numpy()
                cx_pred = preds['center_xy'][pos_mask, 0].cpu().numpy()
                cx_true = batch['center_xy'][pos_mask, 0].numpy()
                cy_pred = preds['center_xy'][pos_mask, 1].cpu().numpy()
                cy_true = batch['center_xy'][pos_mask, 1].numpy()
                type_pred_idx = preds['type_logit'][pos_mask].argmax(dim=1).cpu().numpy()
                type_true_idx = batch['type_idx'][pos_mask].numpy()

                all_t_pred.append(t_pred)
                all_t_true.append(t_true)
                all_cx_pred.append(cx_pred)
                all_cx_true.append(cx_true)
                all_cy_pred.append(cy_pred)
                all_cy_true.append(cy_true)
                all_type_pred.extend([cfg.idx2type[i] for i in type_pred_idx])
                all_type_true.extend([cfg.idx2type[i] for i in type_true_idx])

    mean_loss = total_loss / max(n_batches, 1)

    if all_t_pred:
        score_dict = batch_competition_score(
            np.concatenate(all_t_pred), np.concatenate(all_t_true),
            np.concatenate(all_cx_pred), np.concatenate(all_cx_true),
            np.concatenate(all_cy_pred), np.concatenate(all_cy_true),
            all_type_pred, all_type_true,
            sigma_t=cfg.sigma_t, sigma_s=cfg.sigma_s,
        )
    else:
        score_dict = {'mean_T': 0.0, 'mean_S': 0.0, 'mean_C': 0.0, 'mean_score': 0.0}

    return mean_loss, score_dict


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: Optional[torch.cuda.amp.GradScaler],
    cfg: Config,
) -> Dict[str, float]:
    model.train()
    total = {'total': 0.0, 'present': 0.0, 'time': 0.0, 'spatial': 0.0, 'type': 0.0}
    n_batches = 0

    pbar = tqdm(loader, desc='Train', leave=False)
    for batch in pbar:
        frames = batch['frames'].to(cfg.device, non_blocking=True)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            preds = model(frames)
            loss_dict = compute_loss(preds, batch, cfg)

        if scaler is not None:
            scaler.scale(loss_dict['total']).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict['total'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        for k in total:
            total[k] += loss_dict[k].item()
        n_batches += 1

        pbar.set_postfix(loss=f"{loss_dict['total'].item():.4f}")

    return {k: v / max(n_batches, 1) for k, v in total.items()}


# Setup optimizer with differential learning rates (heads at lr_heads, backbone at lr_backbone)
optimizer = torch.optim.AdamW([
    {'params': model.get_backbone_params(), 'lr': cfg.lr_backbone},
    {'params': model.get_head_params(),     'lr': cfg.lr_heads},
], weight_decay=cfg.weight_decay)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.num_epochs, eta_min=1e-6
)

scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

# Freeze backbone for first freeze_epochs
model.freeze_backbone()
print("Backbone frozen for first", cfg.freeze_epochs, "epochs.")

best_val_score = -1.0
history = []

for epoch in range(1, cfg.num_epochs + 1):
    # Unfreeze backbone after freeze_epochs
    if epoch == cfg.freeze_epochs + 1:
        model.unfreeze_backbone()
        print(f"\n--- Epoch {epoch}: Backbone unfrozen ---")

    train_losses = train_one_epoch(model, train_loader, optimizer, scaler, cfg)
    val_loss, val_scores = evaluate_on_loader(model, val_loader, cfg, scaler)
    scheduler.step()

    val_score = val_scores['mean_score']

    print(
        f"Epoch {epoch:02d}/{cfg.num_epochs} | "
        f"train_loss={train_losses['total']:.4f} "
        f"(pres={train_losses['present']:.3f} "
        f"time={train_losses['time']:.3f} "
        f"spatial={train_losses['spatial']:.3f} "
        f"type={train_losses['type']:.3f}) | "
        f"val_loss={val_loss:.4f} | "
        f"val_score={val_score:.4f} "
        f"(T={val_scores['mean_T']:.3f} "
        f"S={val_scores['mean_S']:.3f} "
        f"C={val_scores['mean_C']:.3f})"
    )

    history.append({
        'epoch': epoch,
        'train_loss': train_losses['total'],
        'val_loss': val_loss,
        'val_score': val_score,
        **{f'val_{k}': v for k, v in val_scores.items()},
    })

    if val_score > best_val_score:
        best_val_score = val_score
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_score': val_score,
            'val_scores': val_scores,
            'cfg': cfg,
        }, cfg.best_ckpt)
        print(f"  ** Saved best model (val_score={val_score:.4f}) **")

print(f"\nTraining complete. Best val competition score: {best_val_score:.4f}")

Backbone frozen for first 3 epochs.


Train:   0%|          | 0/773 [00:00<?, ?it/s]

Epoch 01/15 | train_loss=2.6908 (pres=0.700 time=0.031 spatial=0.024 type=1.275) | val_loss=2.2753 | val_score=0.2913 (T=1.000 S=0.338 C=0.623)
  ** Saved best model (val_score=0.2913) **


Train:   0%|          | 0/773 [00:00<?, ?it/s]

## Training Curve Visualization

- Plotting both **loss curves and competition sub-score curves** side by side allows diagnosing two distinct failure modes: (1) overfitting (train loss decreasing while val loss rises) and (2) metric-loss misalignment (val loss improving while competition score stagnates, which would indicate the loss formulation is not well-aligned with the metric).
- The **per-component curves** (T, S, C, and harmonic mean) reveal which sub-task is the bottleneck. For example, if T and S both plateau near 0.8 but C remains near 0.3, the type classification head is the weak point and may need more weight in the loss function or additional augmentation.
- Saving the figure to `checkpoints/training_curves.png` persists it alongside the model checkpoint, enabling post-hoc analysis without re-running training.
- The **tail of the history DataFrame** is printed to show the final epoch numbers, which is useful for confirming the cosine scheduler reached its minimum and that training did not terminate prematurely.

In [ ]:
# Cell 11: Checkpoint — plot training curves
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', marker='o')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val Loss',   marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(hist_df['epoch'], hist_df['val_score'],    label='Val Score (HM)', marker='o', linewidth=2)
axes[1].plot(hist_df['epoch'], hist_df['val_mean_T'],   label='T (temporal)',  marker='s', linestyle='--')
axes[1].plot(hist_df['epoch'], hist_df['val_mean_S'],   label='S (spatial)',   marker='^', linestyle='--')
axes[1].plot(hist_df['epoch'], hist_df['val_mean_C'],   label='C (type acc.)', marker='D', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Val Competition Metrics per Epoch')
axes[1].legend()
axes[1].grid(True)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(str(cfg.checkpoint_dir / 'training_curves.png'), dpi=100)
plt.show()
print(f"Best val competition score: {best_val_score:.4f}")
print(hist_df.tail())

## Final Evaluation on the Held-Out Synthetic Test Set

- **Loading the best checkpoint** (selected by validation competition score) rather than using the model at the end of training guards against the final epoch being suboptimal due to LR oscillation or noise in the last few batches.
- This cell provides the **synthetic test set performance** — the highest-quality proxy for real leaderboard performance available during development. Because test videos are drawn from the same CARLA simulator distribution as training data, these numbers represent an upper bound on real-world performance.
- **Interpreting the gap between synthetic test score and real leaderboard score** is the key diagnostic for sim-to-real transfer. A large gap (e.g., 0.7 on synthetic vs. 0.3 on real) indicates significant domain adaptation challenges — visual appearance differences (CARLA renders vs. real CCTV), camera angle distributions, and real-world occlusion/blur that the model has never seen during training.
- The `scaler=None` argument to `evaluate_on_loader` disables AMP during test evaluation for numerical precision — AMP's float16 approximations are acceptable for gradient computation but undesirable when computing final metric values.

In [ ]:
# Cell 12: Final evaluation — load best checkpoint, run on test split
ckpt = torch.load(cfg.best_ckpt, map_location=cfg.device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} with val_score={ckpt['val_score']:.4f}")

test_loss, test_scores = evaluate_on_loader(model, test_loader, cfg, scaler=None)

print("\n=== Test Set Results ===")
print(f"Test Loss:        {test_loss:.4f}")
print(f"T (temporal):     {test_scores['mean_T']:.4f}")
print(f"S (spatial):      {test_scores['mean_S']:.4f}")
print(f"C (type acc.):    {test_scores['mean_C']:.4f}")
print(f"Harmonic Mean:    {test_scores['mean_score']:.4f}")

## Detailed Results Analysis: Confusion Matrix and Error Scatter Plots

- **Confusion matrix** on the test split reveals which collision types are confused with one another. Systematic confusions (e.g., `sideswipe` predicted as `t-bone`) point to specific visual similarities in the synthetic domain that the model has not disentangled — and these confusions are likely to persist or worsen on real CCTV footage due to the additional noise of the domain gap.
- **Time error scatter plot** (predicted − true vs. true time offset) diagnoses systematic bias: if the model consistently underestimates or overestimates the offset, the regression head may need a bias correction or a different loss formulation (e.g., Huber loss instead of MSE to reduce sensitivity to outlier clips).
- **Spatial error color map** visualizes whether location prediction errors are uniform across the frame or cluster in specific regions (e.g., frame edges or corners). CARLA cameras are positioned at fixed angles in synthetic videos; real CCTV cameras have widely varying placements, so edge-heavy errors in synthetic evaluation may indicate the backbone has learned camera-specific spatial priors.
- MAE and RMSE for time prediction are printed as secondary metrics — MAE is more interpretable (mean seconds off) while RMSE highlights large outliers.
- The **classification report** (precision, recall, F1 per type) provides finer detail than the confusion matrix alone, especially for rarer collision types.

In [ ]:
# Cell 13: Results summary — confusion matrix + scatter plots

# Re-run inference on test set collecting all predictions
model.eval()

all_t_pred, all_t_true = [], []
all_cx_pred, all_cx_true = [], []
all_cy_pred, all_cy_true = [], []
all_type_pred_str, all_type_true_str = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test inference'):
        frames = batch['frames'].to(cfg.device, non_blocking=True)
        preds = model(frames)

        pos_mask = batch['accident_present'].bool()
        if pos_mask.any():
            all_t_pred.append(preds['time_offset'][pos_mask].cpu().numpy())
            all_t_true.append(batch['time_offset'][pos_mask].numpy())
            all_cx_pred.append(preds['center_xy'][pos_mask, 0].cpu().numpy())
            all_cx_true.append(batch['center_xy'][pos_mask, 0].numpy())
            all_cy_pred.append(preds['center_xy'][pos_mask, 1].cpu().numpy())
            all_cy_true.append(batch['center_xy'][pos_mask, 1].numpy())
            type_pred_idx = preds['type_logit'][pos_mask].argmax(dim=1).cpu().numpy()
            type_true_idx = batch['type_idx'][pos_mask].numpy()
            all_type_pred_str.extend([cfg.idx2type[i] for i in type_pred_idx])
            all_type_true_str.extend([cfg.idx2type[i] for i in type_true_idx])

t_pred_arr  = np.concatenate(all_t_pred)
t_true_arr  = np.concatenate(all_t_true)
cx_pred_arr = np.concatenate(all_cx_pred)
cx_true_arr = np.concatenate(all_cx_true)
cy_pred_arr = np.concatenate(all_cy_pred)
cy_true_arr = np.concatenate(all_cy_true)

print(f"Total positive test clips evaluated: {len(t_pred_arr)}")

# --- Confusion matrix ---
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cm = confusion_matrix(all_type_true_str, all_type_pred_str, labels=cfg.type_classes)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cfg.type_classes)
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix (Test Positive Clips)')
axes[0].tick_params(axis='x', rotation=30)

# --- Time error scatter ---
time_err = t_pred_arr - t_true_arr
axes[1].scatter(t_true_arr, time_err, alpha=0.4, s=10)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('True time offset (s)')
axes[1].set_ylabel('Predicted - True (s)')
axes[1].set_title(f'Time Prediction Error\nMAE={np.abs(time_err).mean():.3f}s, RMSE={np.sqrt((time_err**2).mean()):.3f}s')
axes[1].grid(True)

# --- Spatial error scatter ---
spatial_err = np.sqrt((cx_pred_arr - cx_true_arr)**2 + (cy_pred_arr - cy_true_arr)**2)
axes[2].scatter(cx_true_arr, cy_true_arr, c=spatial_err, cmap='RdYlGn_r', alpha=0.6, s=15)
sc = axes[2].scatter(cx_true_arr, cy_true_arr, c=spatial_err, cmap='RdYlGn_r', alpha=0.6, s=15)
plt.colorbar(sc, ax=axes[2], label='L2 distance error')
axes[2].set_xlabel('True center_x')
axes[2].set_ylabel('True center_y')
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)
axes[2].set_title(f'Spatial Error by Location\nMean L2={spatial_err.mean():.4f}')
axes[2].grid(True)

plt.tight_layout()
plt.savefig(str(cfg.checkpoint_dir / 'test_results.png'), dpi=100)
plt.show()

print("\nClassification Report:")
print(classification_report(all_type_true_str, all_type_pred_str, labels=cfg.type_classes))

## Sliding-Window Inference on Real Test Videos

- **Sliding-window strategy:** The model was trained on fixed 16-frame clips, so at test time we slide this window across the full video with `stride_infer=8` (50% overlap) and collect predictions from every window. The window with the **highest accident presence probability** is selected, and its time offset, spatial location, and type predictions are used.
- **Why 50% overlap (stride=8)?** Halving the stride relative to clip length ensures no accident frame falls entirely outside a window's field of view. With stride=16 (no overlap), an accident at the boundary between two windows might be split across both, causing both to score low presence probability and the accident to be missed. The 2× overhead is worth the improved temporal precision.
- **Absolute time computation:** The model predicts `time_offset` relative to the clip start. Converting to absolute video time requires: `abs_time = (start_frame + time_offset * fps) / fps`. This is clipped to [0, total_frames/fps] to prevent out-of-bounds predictions for edge windows.
- **Fallback prediction** (`rear-end` at t=5.0s, center=(0.5, 0.5)) is returned for unreadable or missing videos. `rear-end` is chosen as the fallback type because it is the most common collision type in the dataset — using the most-frequent class as a default maximizes expected C-score for corrupted videos.
- **Batch processing within a video** (inner loop over `batch_sz` windows at a time) avoids loading all windows into GPU memory simultaneously for long videos, which could cause OOM on videos with hundreds of sliding positions.

In [ ]:
# Cell 14: Sliding-window inference on real test videos (submission generation)
def infer_video_sliding_window(
    model: nn.Module,
    video_path: str,
    cfg: Config,
) -> Dict:
    """
    Run sliding-window inference on a single video.
    Selects the clip with highest accident_present probability.
    Returns dict: {accident_time, center_x, center_y, type}
    """
    model.eval()
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 20.0
    cap.release()

    if total_frames <= 0:
        # Fallback for unreadable video
        return {
            'accident_time': 5.0,
            'center_x': 0.5,
            'center_y': 0.5,
            'type': 'rear-end',
        }

    clip_len = cfg.clip_len
    stride = cfg.stride_infer

    # Generate start frames for all windows
    max_start = max(0, total_frames - clip_len)
    start_frames = list(range(0, max_start + 1, stride))
    if not start_frames:
        start_frames = [0]

    best_prob = -1.0
    best_result = None

    # Process clips in small batches
    batch_sz = cfg.batch_size
    mean_arr = np.array(cfg.mean, dtype=np.float32).reshape(1, 3, 1, 1)
    std_arr  = np.array(cfg.std,  dtype=np.float32).reshape(1, 3, 1, 1)

    def load_clip_frames(video_path: str, start_frame: int) -> np.ndarray:
        cap = cv2.VideoCapture(video_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        frames = []
        last_frame = None
        for _ in range(clip_len):
            ret, frame = cap.read()
            if not ret:
                frames.append(last_frame if last_frame is not None else
                               np.zeros((cfg.img_size, cfg.img_size, 3), dtype=np.uint8))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (cfg.img_size, cfg.img_size), interpolation=cv2.INTER_LINEAR)
                last_frame = frame
                frames.append(frame)
        cap.release()
        arr = np.stack(frames, axis=0).astype(np.float32) / 255.0  # (T, H, W, C)
        arr = arr.transpose(0, 3, 1, 2)  # (T, C, H, W)
        arr = (arr - mean_arr) / std_arr
        arr = arr.transpose(1, 0, 2, 3)  # (C, T, H, W)
        return arr

    for batch_start in range(0, len(start_frames), batch_sz):
        batch_starts = start_frames[batch_start: batch_start + batch_sz]
        clips = np.stack([load_clip_frames(video_path, sf) for sf in batch_starts], axis=0)
        clips_tensor = torch.from_numpy(clips).to(cfg.device)

        with torch.no_grad():
            preds = model(clips_tensor)
            probs = torch.sigmoid(preds['accident_present_logit']).cpu().numpy()

        for i, (sf, prob) in enumerate(zip(batch_starts, probs)):
            if prob > best_prob:
                best_prob = float(prob)
                time_off = float(preds['time_offset'][i].cpu().item())
                time_off = max(0.0, time_off)
                abs_time = (sf + time_off * fps) / fps

                cx = float(preds['center_xy'][i, 0].cpu().item())
                cy = float(preds['center_xy'][i, 1].cpu().item())
                type_idx = int(preds['type_logit'][i].argmax().cpu().item())

                best_result = {
                    'accident_time': float(np.clip(abs_time, 0.0, total_frames / fps)),
                    'center_x': float(np.clip(cx, 0.0, 1.0)),
                    'center_y': float(np.clip(cy, 0.0, 1.0)),
                    'type': cfg.idx2type[type_idx],
                }

    return best_result


# Load test metadata
test_meta = pd.read_csv(cfg.test_meta_csv)
print(f"Test videos: {len(test_meta)}")
print(test_meta.head())

## Generating the Final Submission CSV

- **Reloading the best checkpoint** at the start of this cell is a defensive practice — the kernel state may have been modified by the diagnostic cells (12, 13), and explicitly reloading ensures the submission uses the correct weights rather than whatever state the model is in at the end of the analysis cells.
- **Per-video try/except** wraps each inference call so that a single corrupted or missing video file does not crash the entire submission run. Failed videos fall back to the default prediction and are logged, allowing a post-hoc review of which real CCTV videos were unprocessable.
- **Type distribution and time statistics** are printed after generating all predictions to detect degenerate outputs (e.g., the model predicting `rear-end` for every video, or all accident times clustering at 0s). Such patterns would indicate the backbone failed to generalize to real footage and is outputting its training-domain prior.
- The submission CSV conforms to the required format (`path, accident_time, center_x, center_y, type`) and is saved to `submission_r3d.csv`. This file is ready for direct upload to the Kaggle competition submission page.
- **Sim-to-real consideration:** Because all training labels come from CARLA synthetic data, the model's type classifier has never seen real camera sensor noise, compression artifacts, or CCTV lens distortion. Monitoring C-score on the public leaderboard relative to T and S indicates how much the visual domain gap hurts classification versus localization.

In [ ]:
# Cell 15: Generate submission CSV
ckpt = torch.load(cfg.best_ckpt, map_location=cfg.device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded model from epoch {ckpt['epoch']}")

results = []
failed_videos = []

for _, row in tqdm(test_meta.iterrows(), total=len(test_meta), desc='Inference on test videos'):
    video_path = str(cfg.test_video_dir / row['path'])
    try:
        if not os.path.exists(video_path):
            raise FileNotFoundError(f"Video not found: {video_path}")
        pred = infer_video_sliding_window(model, video_path, cfg)
    except Exception as e:
        failed_videos.append((row['path'], str(e)))
        pred = {
            'accident_time': 5.0,
            'center_x': 0.5,
            'center_y': 0.5,
            'type': 'rear-end',
        }

    results.append({
        'path': row['path'],
        'accident_time': pred['accident_time'],
        'center_x': pred['center_x'],
        'center_y': pred['center_y'],
        'type': pred['type'],
    })

submission_df = pd.DataFrame(results)
submission_df.to_csv(cfg.submission_csv, index=False)

print(f"\nSubmission saved to: {cfg.submission_csv}")
print(f"Shape: {submission_df.shape}")
print(f"Failed videos: {len(failed_videos)}")
if failed_videos:
    for path, err in failed_videos[:5]:
        print(f"  {path}: {err}")
print(f"\nType distribution in submission:")
print(submission_df['type'].value_counts())
print(f"\nAccident time stats:")
print(submission_df['accident_time'].describe())
submission_df.head()